In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]


tau = length(Istar_obs)

model_tag_sym = :memoryless

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [memoryless_model] Fitting chain 2 (tau=34)
[ Info: [memoryless] iter 1000/1000000 elapsed=3.9s, rate=0.207, mean=[1.177, 0.00102, 1.557], std=[0.1122, 0.000548, 0.4403] [ADAPT]
[ Info: [memoryless] iter 2000/1000000 elapsed=6.7s, rate=0.203, mean=[1.192, 0.00113, 2.142], std=[0.0826, 0.000429, 0.6244] [ADAPT]
[ Info: [memoryless] iter 3000/1000000 elapsed=8.9s, rate=0.200, mean=[1.192, 0.00120, 2.341], std=[0.0691, 0.000387, 0.5716] [ADAPT]
[ Info: [memoryless] iter 4000/1000000 elapsed=11.0s, rate=0.207, mean=[1.196, 0.00121, 2.452], std=[0.0617, 0.000349, 0.5284] [ADAPT]
[ Info: [memoryless] iter 5000/1000000 elapsed=13.1s, rate=0.207, mean=[1.195, 0.00125, 2.499], std=[0.0567, 0.000343, 0.4850] [ADAPT]
[ Info: [memoryless] iter 6000/1000000 elapsed=15.3s, rate=0.207, mean=[1.189, 0.00127, 2.493], std=[0.0548, 0.000338, 0.4449] [ADAPT]
[ Info: [memoryless] iter 7000/1000000 elapsed=17.5s, rate=0.211, mean=[1.191, 0.00127, 2.523], std=[0.0523, 0.000324, 0.4185] [ADAPT]
[ Info